# Feature Engineering

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))

import pandas as pd

from stocks.features.technical import (
    add_moving_averages,
    add_returns,
    add_rsi,
    add_volatility,
)
from stocks.utils.io import processed_features_path, raw_prices_path

In [ ]:
FOCUS_TICKERS = ["AAPL", "MSFT", "TSLA", "NVDA", "XRP-USD"]

df = pd.read_parquet(raw_prices_path())
df = df[df.index.get_level_values("ticker").isin(FOCUS_TICKERS)]

print("Raw shape:", df.shape)
print("Tickers:", sorted(df.index.get_level_values("ticker").unique()))

In [ ]:
frames = []

for ticker in FOCUS_TICKERS:
    sub = df.xs(ticker, level="ticker").copy()
    sub = sub.dropna(subset=["close"])

    sub = add_returns(sub)
    sub = add_volatility(sub, window=20)
    sub = add_moving_averages(sub, windows=[10, 20, 50])
    sub = add_rsi(sub, period=14)
    sub["target_direction"] = (sub["return"].shift(-1) > 0).astype(int)
    sub["ticker"] = ticker
    frames.append(sub)

features = pd.concat(frames)
features = features.reset_index().rename(columns={"Date": "date"})

feature_cols = [
    "return", "log_return", "vol_20d",
    "sma_10", "sma_20", "sma_50", "rsi_14",
    "volume",
]
features = features.dropna(subset=feature_cols + ["target_direction"])

print("Feature table shape:", features.shape)
features.head()

In [ ]:
# Class balance per ticker
balance = (
    features.groupby("ticker")["target_direction"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .rename(columns={0: "pct_down", 1: "pct_up"})
)
print("Target balance (share of up/down days):")
balance

In [ ]:
out_path = processed_features_path()
features.to_parquet(out_path, compression="zstd", index=False)
print(f"Saved -> {out_path}")
print(f"Rows: {len(features)}, Columns: {len(features.columns)}")